## 1. Imports

In [1]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

SEARCH_PATH = os.path.join(
    PROCESSED_PATH,
    "search"
)

PROFILE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_profiles"
)

TIMELINE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_timeline"
)

ANALYTICS_PATH = os.path.join(
    PROCESSED_PATH,
    "company_analytics"
)

GRAPH_PATH = os.path.join(
    PROCESSED_PATH,
    "knowledge_graph"
)

FINANCIAL_DATASET = os.path.join(
    PROCESSED_PATH,
    "financial_intelligence",
    "financial_intelligence_dataset.parquet"
)

os.makedirs(
    SEARCH_PATH,
    exist_ok=True
)

## 3. Load Datasets

In [3]:
financial_df = pd.read_parquet(
    FINANCIAL_DATASET
)

company_profiles = pd.read_parquet(
    os.path.join(
        PROFILE_PATH,
        "company_profiles.parquet"
    )
)

timeline_df = pd.read_parquet(
    os.path.join(
        TIMELINE_PATH,
        "company_event_timeline.parquet"
    )
)

analytics = pd.read_parquet(
    os.path.join(
        ANALYTICS_PATH,
        "company_analytics.parquet"
    )
)

company_similarity = pd.read_parquet(
    os.path.join(
        ANALYTICS_PATH,
        "company_similarity.parquet"
    )
)

nodes = pd.read_parquet(
    os.path.join(
        GRAPH_PATH,
        "nodes.parquet"
    )
)

edges = pd.read_parquet(
    os.path.join(
        GRAPH_PATH,
        "edges.parquet"
    )
)

company_neighbors = pd.read_parquet(
    os.path.join(
        GRAPH_PATH,
        "company_neighbors.parquet"
    )
)

print("Financial Dataset")
print(financial_df.shape)

print()

print("Company Profiles")
print(company_profiles.shape)

print()

print("Timeline")
print(timeline_df.shape)

print()

print("Analytics")
print(analytics.shape)

print()

print("Similarity")
print(company_similarity.shape)

print()

print("Knowledge Graph Nodes")
print(nodes.shape)

print()

print("Knowledge Graph Edges")
print(edges.shape)

print()

print("Company Neighbors")
print(company_neighbors.shape)

Financial Dataset
(3215296, 22)

Company Profiles
(6619, 60)

Timeline
(3215296, 26)

Analytics
(6619, 70)

Similarity
(66190, 3)

Knowledge Graph Nodes
(7720, 18)

Knowledge Graph Edges
(427023, 5)

Company Neighbors
(6619, 3)


## 4. Build Search Indexes

In [4]:
# ---------- Company ----------

company_index = {

    str(row["ticker"]).upper(): idx

    for idx, row in company_profiles.iterrows()

}

# ---------- Publisher ----------

publisher_index = {

    str(pub).lower(): pub

    for pub in financial_df["publisher"].dropna().unique()

}

# ---------- Event ----------

event_index = {

    str(event).lower(): event

    for event in financial_df["final_event"].dropna().unique()

}

# ---------- Cluster ----------

cluster_index = {

    int(cluster): cluster

    for cluster in analytics["cluster"].unique()

}

# ---------- Fingerprint ----------

fingerprint_index = {

    fp.lower(): fp

    for fp in analytics["company_fingerprint"].dropna().unique()

}

# ---------- News ID ----------

news_index = {

    int(row["news_id"]): idx

    for idx, row in financial_df.iterrows()

}

print("Search Index Statistics")

print(f"Companies      : {len(company_index):,}")
print(f"Publishers     : {len(publisher_index):,}")
print(f"Events         : {len(event_index):,}")
print(f"Clusters       : {len(cluster_index):,}")
print(f"Fingerprints   : {len(fingerprint_index):,}")
print(f"News IDs       : {len(news_index):,}")

Search Index Statistics
Companies      : 6,619
Publishers     : 1,047
Events         : 30
Clusters       : 12
Fingerprints   : 8
News IDs       : 3,215,296


## 5. Build Lookup Tables

In [5]:
company_lookup = {

    str(row["ticker"]).upper(): row.to_dict()

    for _, row in company_profiles.iterrows()

}

analytics_lookup = {

    str(row["ticker"]).upper(): row.to_dict()

    for _, row in analytics.iterrows()

}

neighbor_lookup = {

    str(row["ticker"]).upper(): row["neighbors"]

    for _, row in company_neighbors.iterrows()

}

timeline_lookup = {

    ticker: group

    for ticker, group in timeline_df.groupby("ticker")

}

similarity_lookup = {

    ticker: group

    for ticker, group in company_similarity.groupby("ticker")

}

print("Lookup Tables Ready")

print(f"Company Profiles   : {len(company_lookup):,}")
print(f"Analytics          : {len(analytics_lookup):,}")
print(f"Timelines          : {len(timeline_lookup):,}")
print(f"Similarity         : {len(similarity_lookup):,}")
print(f"Neighbors          : {len(neighbor_lookup):,}")

Lookup Tables Ready
Company Profiles   : 6,619
Analytics          : 6,619
Timelines          : 6,619
Similarity         : 6,619
Neighbors          : 6,619


## 6. Quick Validation

In [6]:
sample_company = "AAPL"

print("Validation")

print("Exists in Profiles  :", sample_company in company_lookup)
print("Exists in Analytics :", sample_company in analytics_lookup)
print("Exists in Timeline  :", sample_company in timeline_lookup)
print("Exists in Similarity:", sample_company in similarity_lookup)
print("Exists in Neighbor  :", sample_company in neighbor_lookup)

Validation
Exists in Profiles  : True
Exists in Analytics : True
Exists in Timeline  : True
Exists in Similarity: True
Exists in Neighbor  : True


## 7. Company Search

In [7]:
def search_company(ticker, timeline_limit=10):

    ticker = str(ticker).upper()

    if ticker not in company_lookup:

        print(f"{ticker} not found.")

        return None

    result = {}

    # Company Profile
    result["profile"] = company_lookup[ticker]

    # Analytics
    result["analytics"] = analytics_lookup[ticker]

    # Timeline
    timeline = timeline_lookup[ticker]

    timeline = (

        timeline

        .sort_values(

            "published_date",

            ascending=False

        )

        .head(timeline_limit)

    )

    result["timeline"] = timeline

    # Similar Companies
    result["similar_companies"] = similarity_lookup[ticker]

    # Graph Neighbors
    result["neighbors"] = neighbor_lookup[ticker]

    return result

## 8. Pretty Print Company Search

In [8]:
def display_company(result):

    if result is None:

        return


    print("COMPANY PROFILE")


    profile = pd.DataFrame([result["profile"]])

    display(profile)

    print()


    print("COMPANY ANALYTICS")


    analytics = pd.DataFrame([result["analytics"]])

    display(analytics)

    print()


    print("SIMILAR COMPANIES")


    display(result["similar_companies"])

    print()


    print("GRAPH NEIGHBORS")

    print(result["neighbors"])

    print()

    print("LATEST TIMELINE")

    display(result["timeline"])

## 9. Test Company Search

In [9]:
result = search_company("TSLA")

display_company(result)

COMPANY PROFILE


,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,Partnership,Product Launch,Regulatory Approval,Revenue Guidance,Share Buyback,Stock Split,Technical Analysis,Trading Halt,risk_rank,intelligence_rank
0,TSLA,1874,2019-07-01 00:00:00+00:00,2020-06-10 21:02:47+00:00,58,87.258271,13.965848,0.818553,0.7612,0.333,...,11,824,15,15,0,84,19,1,192,712



COMPANY ANALYTICS


,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster,company_fingerprint,risk_explanation
0,TSLA,1874,2019-07-01 00:00:00+00:00,2020-06-10 21:02:47+00:00,58,87.258271,13.965848,0.818553,0.7612,0.333,...,40.29,Medium,28,93.28,66.37,79.57,12,4,"Bullish, Diversified",Fraud: 5 | Cybersecurity: 2 | Litigation: 19 |...



SIMILAR COMPANIES


,ticker,similar_company,similarity_score
110,TSLA,NAV,0.9461
111,TSLA,NXPI,0.9433
112,TSLA,TTM,0.9417
113,TSLA,NOK,0.9357
114,TSLA,UA,0.9349
115,TSLA,HMC,0.9348
116,TSLA,TM,0.9336
117,TSLA,NE,0.9333
118,TSLA,W,0.9326
119,TSLA,KBH,0.9315



GRAPH NEIGHBORS
['NAV' 'NXPI' 'TTM' 'NOK' 'UA' 'HMC' 'TM' 'NE' 'W' 'KBH']

LATEST TIMELINE


,news_id,ticker,published_date,headline,publisher,final_event,market_signal,final_confidence,event_number,previous_event_date,...,bullish_event,bearish_event,bullish_events_so_far,bearish_events_so_far,event_importance,event_severity,event_date,daily_event_count,activity_level,is_latest_event
2857678,1245072,TSLA,2020-06-10 21:02:47+00:00,Tesla's Stock Closes At All-Time High As Musk ...,Drew Levine,Stock Split,Neutral,0.7389,1874,2020-06-10 19:08:09+00:00,...,0,0,157,49,4,Medium,2020-06-10,10,High,True
2857677,1245073,TSLA,2020-06-10 19:08:09+00:00,'Tesla factory workplace safety is 5% better t...,Benzinga Newsdesk,Product Launch,Neutral,0.7136,1873,2020-06-10 16:41:58+00:00,...,0,0,157,49,7,High,2020-06-10,10,High,False
2857676,1245074,TSLA,2020-06-10 16:41:58+00:00,'Tesla hacker unlocks Performance upgrade and ...,Benzinga Newsdesk,Analyst Rating,Bullish,1.0000,1872,2020-06-10 15:33:18+00:00,...,1,0,157,49,5,Medium,2020-06-10,10,High,False
2857675,1245075,TSLA,2020-06-10 15:33:18+00:00,GM On Track To Spend $20B On EV And AV Develop...,Benzinga Newsdesk,Product Launch,Neutral,0.7237,1871,2020-06-10 14:15:07+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857674,1245076,TSLA,2020-06-10 14:15:07+00:00,"Tesla's Journey To $1,000 In 2020",Wayne Duggan,Product Launch,Neutral,0.7711,1870,2020-06-10 13:58:54+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857673,1245077,TSLA,2020-06-10 13:58:54+00:00,Tesla Shares Mark Session And New All-Time Hig...,Benzinga Newsdesk,Product Launch,Neutral,0.7460,1869,2020-06-10 13:00:24+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857672,1245078,TSLA,2020-06-10 13:00:24+00:00,Wedbush Says Tesla Has 'Game Changing' Develop...,Jayson Derrick,Product Launch,Neutral,0.7269,1868,2020-06-10 12:43:13+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857671,1245079,TSLA,2020-06-10 12:43:13+00:00,"Wedbush Maintains Neutral on Tesla, Raises Pri...",Benzinga Newsdesk,Analyst Rating,Bullish,1.0000,1867,2020-06-10 12:14:33+00:00,...,1,0,156,49,5,Medium,2020-06-10,10,High,False
2857670,1245080,TSLA,2020-06-10 12:14:33+00:00,Musk Says It's Time To Ramp Production Of Tesl...,Tanzeel Akhtar,Product Launch,Neutral,0.7728,1866,2020-06-10 11:55:49+00:00,...,0,0,155,49,7,High,2020-06-10,10,High,False
2857669,1245081,TSLA,2020-06-10 11:55:49+00:00,Tesla shares are trading higher after Wedbush ...,Benzinga Newsdesk,Analyst Rating,Bullish,0.8000,1865,2020-06-02 00:00:00+00:00,...,1,0,155,49,5,Medium,2020-06-10,10,High,False


## 10. Publisher Search

In [10]:
def search_publisher(publisher_name, limit=20):

    publisher_name = str(publisher_name).lower()

    publishers = financial_df["publisher"].astype(str)

    mask = publishers.str.lower() == publisher_name

    results = financial_df.loc[mask].copy()

    if len(results) == 0:

        print(f"{publisher_name} not found.")

        return None

    results = results.sort_values(

        "published_date",

        ascending=False

    )

    summary = {

        "publisher": publisher_name,

        "total_articles": len(results),

        "companies": results["ticker"].nunique(),

        "events": results["final_event"].nunique(),

        "avg_confidence": round(

            results["final_confidence"].mean(),

            4

        )

    }

    return summary, results.head(limit)

In [11]:
summary, articles = search_publisher(

    "Benzinga Newsdesk"

)

print(summary)

display(articles)

{'publisher': 'benzinga newsdesk', 'total_articles': 150392, 'companies': 4049, 'events': 30, 'avg_confidence': np.float64(0.8482)}


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
972643,972647,PG&E Corp Reports CPUC Approved Co.'s Microgri...,https://www.benzinga.com/news/20/06/16238398/p...,Benzinga Newsdesk,2020-06-11 21:11:20+00:00,PCG,analyst,2020,6,11,...,57,8,True,PG&E Corp Reports CPUC Approved Co.'s Microgri...,57,8,Regulatory Approval,Bullish,1.0000,Rule
1258838,1258842,"Twitter Removes About 174,000 China-Linked Acc...",https://www.benzinga.com/tech/20/06/16238284/t...,Benzinga Newsdesk,2020-06-11 21:01:39+00:00,TWTR,analyst,2020,6,11,...,130,18,True,"Twitter Removes About 174,000 China-Linked Acc...",130,18,Other,Neutral,0.6933,Unknown
1347859,1347863,W. P. Carey Inc. Increases Quarterly Dividend ...,https://www.benzinga.com/news/20/06/16237774/w...,Benzinga Newsdesk,2020-06-11 20:30:31+00:00,WPC,analyst,2020,6,11,...,61,9,True,W. P. Carey Inc. Increases Quarterly Dividend ...,61,9,Dividend,Bullish,0.6670,Rule
1030723,1030727,PVH shares are trading lower after the company...,https://www.benzinga.com/wiim/20/06/16237694/p...,Benzinga Newsdesk,2020-06-11 20:25:21+00:00,PVH,analyst,2020,6,11,...,101,15,True,PVH shares are trading lower after the company...,101,15,Earnings,Bearish,0.8330,Rule
1030725,1030729,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Newsdesk,2020-06-11 20:15:38+00:00,PVH,analyst,2020,6,11,...,82,13,True,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,82,13,Earnings,Bearish,1.0000,Rule
1193973,1193977,UPDATE: Feinseth Says 'You also have Comcast w...,https://www.benzinga.com/analyst-ratings/analy...,Benzinga Newsdesk,2020-06-11 18:22:24+00:00,T,analyst,2020,6,11,...,214,36,True,UPDATE: Feinseth Says 'You also have Comcast w...,214,36,Other,Neutral,0.6387,Unknown
1193974,1193978,UPDATE: Feinseth Says Believes Disney+ Will Be...,https://www.benzinga.com/analyst-ratings/analy...,Benzinga Newsdesk,2020-06-11 18:22:20+00:00,T,analyst,2020,6,11,...,195,30,True,UPDATE: Feinseth Says Believes Disney+ Will Be...,195,30,Other,Neutral,0.6863,Unknown
1193975,1193979,UPDATE: Tigress's Feinseth Says 'I think that ...,https://www.benzinga.com/analyst-ratings/analy...,Benzinga Newsdesk,2020-06-11 18:22:16+00:00,T,analyst,2020,6,11,...,252,43,True,UPDATE: Tigress's Feinseth Says 'I think that ...,252,43,Other,Neutral,0.6684,Unknown
1193976,1193980,Tigress Financial Analyst Ivan Feinseth On Fil...,https://www.benzinga.com/analyst-ratings/analy...,Benzinga Newsdesk,2020-06-11 18:22:11+00:00,T,analyst,2020,6,11,...,240,37,True,Tigress Financial Analyst Ivan Feinseth On Fil...,240,37,Other,Neutral,0.6382,Unknown
1384597,1384601,Yum! Brands Files Lawsuit Against GrubHub Rela...,https://www.benzinga.com/news/20/06/16235934/y...,Benzinga Newsdesk,2020-06-11 17:37:48+00:00,YUM,analyst,2020,6,11,...,73,10,True,Yum! Brands Files Lawsuit Against GrubHub Rela...,73,10,Litigation,Bearish,0.8000,Rule


## 11. Event Search

In [12]:
def search_event(event_name, limit=20):

    event_name = str(event_name).lower()

    mask = (

        financial_df["final_event"]

        .astype(str)

        .str.lower()

        == event_name

    )

    results = financial_df.loc[mask].copy()

    if len(results) == 0:

        print(f"{event_name} not found.")

        return None

    results = results.sort_values(

        "published_date",

        ascending=False

    )

    summary = {

        "event": event_name,

        "articles": len(results),

        "companies": results["ticker"].nunique(),

        "publishers": results["publisher"].nunique(),

        "avg_confidence": round(

            results["final_confidence"].mean(),

            4

        )

    }

    return summary, results.head(limit)

In [13]:
summary, articles = search_event(

    "Earnings"

)

print(summary)

display(articles)

{'event': 'earnings', 'articles': 567435, 'companies': 5694, 'publishers': 396, 'avg_confidence': np.float64(0.9441)}


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
1030723,1030727,PVH shares are trading lower after the company...,https://www.benzinga.com/wiim/20/06/16237694/p...,Benzinga Newsdesk,2020-06-11 20:25:21+00:00,PVH,analyst,2020,6,11,...,101,15,True,PVH shares are trading lower after the company...,101,15,Earnings,Bearish,0.833,Rule
1030724,1030728,PVH: Q1 Earnings Insights,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-11 20:24:41+00:00,PVH,analyst,2020,6,11,...,25,4,True,PVH: Q1 Earnings Insights,25,4,Earnings,Neutral,1.000,Rule
1030725,1030729,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Newsdesk,2020-06-11 20:15:38+00:00,PVH,analyst,2020,6,11,...,82,13,True,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,82,13,Earnings,Bearish,1.000,Rule
1332872,1332876,A Look Into Wells Fargo's Price Over Earnings,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:41:22+00:00,WFC,analyst,2020,6,11,...,45,8,True,A Look Into Wells Fargo's Price Over Earnings,45,8,Earnings,Neutral,1.000,Rule
846190,846193,Price Over Earnings Overview: Morgan Stanley,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:03:37+00:00,MS,analyst,2020,6,11,...,44,6,True,Price Over Earnings Overview: Morgan Stanley,44,6,Earnings,Neutral,1.000,Rule
1298751,1298755,Shares of several airline companies are tradin...,https://www.benzinga.com/wiim/20/06/16232916/s...,Benzinga Newsdesk,2020-06-11 14:02:29+00:00,VLRS,analyst,2020,6,11,...,189,30,True,Shares of several airline companies are tradin...,189,30,Earnings,Bearish,0.444,Rule
776340,776343,Shares of several airline companies are tradin...,https://www.benzinga.com/wiim/20/06/16232916/s...,Benzinga Newsdesk,2020-06-11 14:02:29+00:00,LUV,analyst,2020,6,11,...,189,30,True,Shares of several airline companies are tradin...,189,30,Earnings,Bearish,0.444,Rule
1267933,1267937,Shares of several airline companies are tradin...,https://www.benzinga.com/wiim/20/06/16232916/s...,Benzinga Newsdesk,2020-06-11 14:02:29+00:00,UAL,analyst,2020,6,11,...,189,30,True,Shares of several airline companies are tradin...,189,30,Earnings,Bearish,0.444,Rule
559441,559444,Shares of several airline companies are tradin...,https://www.benzinga.com/wiim/20/06/16232916/s...,Benzinga Newsdesk,2020-06-11 14:02:29+00:00,GOL,analyst,2020,6,11,...,189,30,True,Shares of several airline companies are tradin...,189,30,Earnings,Bearish,0.444,Rule
1110044,1110048,ServiceMaster Sees Q2 Adj. EBITDA From Continu...,https://www.benzinga.com/news/20/06/16230728/s...,Benzinga Newsdesk,2020-06-11 11:32:37+00:00,SERV,analyst,2020,6,11,...,109,14,True,ServiceMaster Sees Q2 Adj. EBITDA From Continu...,109,14,Earnings,Neutral,1.000,Rule


## 12. Timeline Search

In [14]:
def search_timeline(ticker):

    ticker = str(ticker).upper()

    if ticker not in timeline_lookup:

        print(f"{ticker} not found.")

        return None

    timeline = (

        timeline_lookup[ticker]

        .sort_values(

            "published_date"

        )

    )

    return timeline

In [15]:
timeline = search_timeline(

    "TSLA"

)

display(timeline.head(20))

,news_id,ticker,published_date,headline,publisher,final_event,market_signal,final_confidence,event_number,previous_event_date,...,bullish_event,bearish_event,bullish_events_so_far,bearish_events_so_far,event_importance,event_severity,event_date,daily_event_count,activity_level,is_latest_event
2855805,1246942,TSLA,2019-07-01 00:00:00+00:00,Tesla's Q2 Delivery Number Could Cause A Big Move,Wayne Duggan,Product Launch,Neutral,0.7681,1,NaT,...,0,0,0,0,7,High,2019-07-01,4,Low,False
2855806,1246943,TSLA,2019-07-01 00:00:00+00:00,'Tesla Electric Airplane? Elon Musk sees elect...,Benzinga Newsdesk,Product Launch,Neutral,0.7710,2,2019-07-01 00:00:00+00:00,...,0,0,0,0,7,High,2019-07-01,4,Low,False
2855807,1246944,TSLA,2019-07-01 00:00:00+00:00,"UPDATE: JMP Reiterates Outperform, $347 Target...",Benzinga_Newsdesk,Analyst Rating,Bullish,1.0000,3,2019-07-01 00:00:00+00:00,...,1,0,1,0,5,Medium,2019-07-01,4,Low,False
2855808,1246945,TSLA,2019-07-01 00:00:00+00:00,Tesla shares are trading higher after JMP Secu...,Hal Lindon,Analyst Rating,Bullish,0.7140,4,2019-07-01 00:00:00+00:00,...,1,0,2,0,5,Medium,2019-07-01,4,Low,False
2855820,1246941,TSLA,2019-07-02 00:00:00+00:00,Electrek.Co Tweet: Tesla's head of Europe is out,Charles Gross,Product Launch,Neutral,0.7212,16,2019-07-02 00:00:00+00:00,...,0,0,2,0,7,High,2019-07-02,12,High,False
2855819,1246940,TSLA,2019-07-02 00:00:00+00:00,'Tesla loses engineering VP amid end-of-quarte...,Benzinga Newsdesk,Insider Trading,Neutral,1.0000,15,2019-07-02 00:00:00+00:00,...,0,0,2,0,4,Medium,2019-07-02,12,High,False
2855818,1246939,TSLA,2019-07-02 00:00:00+00:00,Reports: 2 More Tesla Execs Depart,Tanzeel Akhtar,Stock Split,Neutral,0.7440,14,2019-07-02 00:00:00+00:00,...,0,0,2,0,4,Medium,2019-07-02,12,High,False
2855817,1246938,TSLA,2019-07-02 00:00:00+00:00,"With Jobs Report Ahead, Focus Turns To Economi...",JJ Kinahan,Product Launch,Neutral,0.8052,13,2019-07-02 00:00:00+00:00,...,0,0,2,0,7,High,2019-07-02,12,High,False
2855816,1246937,TSLA,2019-07-02 00:00:00+00:00,Benzinga Pro's Top 10 Most-Searched Tickers Fo...,Benzinga Newsdesk,Other,Neutral,0.6980,12,2019-07-02 00:00:00+00:00,...,0,0,2,0,1,Low,2019-07-02,12,High,False
2855815,1246936,TSLA,2019-07-02 00:00:00+00:00,A Look At Benzinga Pro's Most-Searched Tickers...,Sebastian Brown,Market Movement,Neutral,0.7574,11,2019-07-02 00:00:00+00:00,...,0,0,2,0,3,Low,2019-07-02,12,High,False


## 13. Similar Company Search

In [16]:
def search_similar_companies(ticker):

    ticker = str(ticker).upper()

    if ticker not in similarity_lookup:

        print(f"{ticker} not found.")

        return None

    return similarity_lookup[ticker]

In [17]:
display(

    search_similar_companies(

        "TSLA"

    )

)

,ticker,similar_company,similarity_score
110,TSLA,NAV,0.9461
111,TSLA,NXPI,0.9433
112,TSLA,TTM,0.9417
113,TSLA,NOK,0.9357
114,TSLA,UA,0.9349
115,TSLA,HMC,0.9348
116,TSLA,TM,0.9336
117,TSLA,NE,0.9333
118,TSLA,W,0.9326
119,TSLA,KBH,0.9315


## 14. Graph Neighbor Search

In [18]:
def search_neighbors(ticker):

    ticker = str(ticker).upper()

    source = "COMP_" + ticker

    neighbors = edges.loc[
        edges["source"] == source
    ].copy()

    if len(neighbors) == 0:

        print(f"No graph neighbors found for {ticker}")

        return None

    return neighbors.sort_values(
        "weight",
        ascending=False
    )

In [19]:
display(
    search_neighbors("TSLA").head(20)
)

,source,target,relation,weight,confidence
337564,COMP_TSLA,SIGNAL_Neutral,HAS_SIGNAL,1640.0,0.807240
306544,COMP_TSLA,EVENT_Product_Launch,HAS_EVENT,824.0,0.761075
181172,COMP_TSLA,PUB_Benzinga_Newsdesk,PUBLISHED_BY,641.0,0.789222
306538,COMP_TSLA,EVENT_Market_Movement,HAS_EVENT,245.0,0.933247
306522,COMP_TSLA,EVENT_Analyst_Rating,HAS_EVENT,228.0,0.930768
181179,COMP_TSLA,PUB_Charles_Gross,PUBLISHED_BY,188.0,0.779358
181200,COMP_TSLA,PUB_Lisa_Levin,PUBLISHED_BY,172.0,0.962095
337562,COMP_TSLA,SIGNAL_Bullish,HAS_SIGNAL,157.0,0.892962
306529,COMP_TSLA,EVENT_Earnings,HAS_EVENT,132.0,0.883417
306542,COMP_TSLA,EVENT_Other,HAS_EVENT,112.0,0.669291


## 15. Cluster Search

In [20]:
def search_cluster(cluster_id):

    cluster_id = int(cluster_id)

    companies = analytics.loc[
        analytics["cluster"] == cluster_id
    ].copy()

    companies = companies.sort_values(
        "composite_score",
        ascending=False
    )

    summary = {

        "cluster": cluster_id,

        "companies": len(companies),

        "avg_score": round(
            companies["composite_score"].mean(),
            2
        ),

        "avg_risk": round(
            companies["normalized_risk_score"].mean(),
            2
        )

    }

    return summary, companies

In [21]:
summary, companies = search_cluster(4)

print(summary)

display(companies.head())

{'cluster': 4, 'companies': 430, 'avg_score': np.float64(70.17), 'avg_risk': np.float64(33.72)}


,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster,company_fingerprint,risk_explanation
11,TSLA,1874,2019-07-01 00:00:00+00:00,2020-06-10 21:02:47+00:00,58,87.258271,13.965848,0.818553,0.7612,0.333,...,40.29,Medium,28,93.28,66.37,79.57,12,4,"Bullish, Diversified",Fraud: 5 | Cybersecurity: 2 | Litigation: 19 |...
19,GOOGL,1751,2018-07-25 00:00:00+00:00,2020-06-10 19:25:13+00:00,75,85.920617,13.331810,0.821730,0.7590,0.375,...,56.44,Medium,30,93.60,62.78,78.02,20,4,"Bullish, Diversified",Fraud: 41 | Bankruptcy: 1 | Cybersecurity: 26 ...
59,USB,1943,2009-08-09 00:00:00+00:00,2020-06-11 14:42:48+00:00,125,67.888832,10.851261,0.849904,0.7825,0.250,...,29.26,Low,30,93.61,43.73,75.59,54,4,"Bullish, Diversified",Fraud: 2 | Bankruptcy: 2 | Cybersecurity: 5 | ...
65,CSCO,1513,2016-08-17 00:00:00+00:00,2020-06-10 13:36:24+00:00,69,77.431593,12.265697,0.836003,0.7693,0.333,...,34.17,Low,28,95.18,47.76,75.43,59,4,"Bullish, Diversified",Fraud: 3 | Cybersecurity: 12 | Litigation: 4 |...
72,AZN,1963,2009-08-30 00:00:00+00:00,2020-06-11 04:16:21+00:00,116,85.127356,12.290881,0.842769,0.7743,0.200,...,25.19,Low,29,94.24,42.68,75.20,64,4,"Bullish, Diversified",Fraud: 2 | Cybersecurity: 1 | Litigation: 19 |...


## 16. Risk Search

In [22]:
def search_high_risk(limit=20):

    results = analytics.sort_values(

        "normalized_risk_score",

        ascending=False

    )

    return results.head(limit)

In [23]:
display(

    search_high_risk()

)

,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster,company_fingerprint,risk_explanation
876,PCG,3314,2009-07-27 00:00:00+00:00,2020-06-11 21:11:20+00:00,93,74.999396,12.073325,0.832868,0.75505,0.333,...,100.00,Very High,29,86.35,48.13,67.18,526,3,"High Risk, Bullish, Diversified",Fraud: 21 | Bankruptcy: 161 | Cybersecurity: 8...
1019,QQQ,2778,2011-03-16 00:00:00+00:00,2020-06-10 16:12:25+00:00,110,75.961843,13.086753,0.897198,1.00000,0.200,...,98.39,Very High,26,98.05,36.10,66.45,590,11,"High Risk, Bullish, Diversified",Cybersecurity: 6 | Litigation: 2 | Layoffs: 17
2573,IWM,1127,2009-08-17 00:00:00+00:00,2020-06-11 13:47:55+00:00,91,101.890861,18.222715,0.895202,1.00000,0.333,...,91.82,Very High,22,98.23,25.58,61.95,1021,2,"High Risk, Bullish, Diversified",Cybersecurity: 3 | Layoffs: 3
617,PANW,3549,2012-07-09 00:00:00+00:00,2020-06-11 18:26:26+00:00,101,66.111299,10.513384,0.835952,0.76560,0.333,...,90.63,Very High,28,93.97,45.12,68.68,409,3,"High Risk, Bullish, Diversified",Fraud: 1 | Cybersecurity: 338 | Litigation: 14...
1317,LL,2306,2009-08-07 00:00:00+00:00,2020-06-11 14:34:01+00:00,106,66.750217,10.235906,0.825449,0.75830,0.333,...,88.85,Very High,29,91.98,34.42,65.44,684,3,"High Risk, Bullish, Diversified",Fraud: 18 | Bankruptcy: 29 | Cybersecurity: 1 ...
2182,CYBR,1792,2014-09-23 00:00:00+00:00,2020-05-15 00:00:00+00:00,65,63.630580,9.846540,0.829998,0.76330,0.333,...,84.54,Very High,26,92.68,28.69,62.92,924,4,"High Risk, Bullish, Diversified",Bankruptcy: 1 | Cybersecurity: 305 | Litigatio...
871,SHLD,2519,2009-08-10 00:00:00+00:00,2019-06-26 00:00:00+00:00,156,62.385073,10.007146,0.806564,0.74720,0.333,...,76.63,High,30,92.54,35.34,67.21,524,3,"High Risk, Bullish, Diversified",Fraud: 1 | Bankruptcy: 59 | Cybersecurity: 14 ...
1992,DIA,980,2018-09-13 00:00:00+00:00,2020-06-10 16:12:25+00:00,25,106.944898,18.783673,0.887796,1.00000,0.250,...,75.08,High,24,96.02,23.73,63.37,881,11,"High Risk, Bullish, Diversified",Fraud: 1 | Bankruptcy: 1 | Cybersecurity: 1 | ...
1061,RBS,1730,2009-08-10 00:00:00+00:00,2020-06-11 14:05:13+00:00,98,64.460116,10.598844,0.833050,0.75895,0.250,...,74.40,High,29,90.69,33.02,66.30,603,4,"High Risk, Bullish, Diversified",Fraud: 41 | Bankruptcy: 2 | Cybersecurity: 16 ...
558,PBR,3597,2009-09-03 00:00:00+00:00,2020-06-11 13:54:38+00:00,106,66.226578,10.448429,0.812073,0.74810,0.250,...,73.09,High,30,89.74,42.19,69.02,380,3,"High Risk, Bullish, Diversified",Fraud: 82 | Bankruptcy: 2 | Cybersecurity: 10 ...


## 17. Fingerprint Search

In [24]:
def search_fingerprint(text):

    text = text.lower()

    results = analytics[

        analytics["company_fingerprint"]

        .str.lower()

        .str.contains(text)

    ].copy()

    return results.sort_values(

        "composite_score",

        ascending=False

    )

In [25]:
display(

    search_fingerprint(

        "Bullish"

    )

)

,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster,company_fingerprint,risk_explanation
0,GILD,4938,2009-09-11 00:00:00+00:00,2020-06-10 17:44:51+00:00,169,75.379911,11.686513,0.820607,0.7535,0.286,...,10.29,Very Low,30,96.41,83.09,89.41,1,3,"Stable, High Activity, Bullish, Diversified",Fraud: 3 | Bankruptcy: 1 | Cybersecurity: 8 | ...
1,HD,4415,2009-08-10 00:00:00+00:00,2020-06-10 12:14:08+00:00,232,64.542922,10.525481,0.829553,0.7599,0.333,...,29.29,Low,30,96.08,74.00,84.88,2,3,"High Activity, Bullish, Diversified",Fraud: 6 | Bankruptcy: 3 | Cybersecurity: 51 |...
2,MDT,4421,2009-08-07 00:00:00+00:00,2020-06-11 14:22:31+00:00,134,68.356254,10.373445,0.851852,0.7805,0.333,...,0.00,Very Low,30,97.71,57.43,83.69,3,3,"Stable, Bullish, Diversified",Fraud: 2 | Bankruptcy: 1 | Cybersecurity: 9 | ...
3,SLV,3968,2009-08-17 00:00:00+00:00,2020-06-03 00:00:00+00:00,119,50.743196,8.517137,0.849060,0.7861,0.250,...,41.21,Medium,28,92.37,72.31,81.65,4,11,"High Activity, Bullish, Diversified",Fraud: 1 | Bankruptcy: 4 | Cybersecurity: 8 | ...
4,LOW,3212,2009-08-13 00:00:00+00:00,2020-06-11 14:34:01+00:00,189,62.575965,10.007161,0.843545,0.7645,0.357,...,23.55,Low,29,96.09,59.47,80.88,5,3,"Bullish, Diversified",Bankruptcy: 6 | Cybersecurity: 9 | Litigation:...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6611,TCCA,2,2018-04-04 10:06:56+00:00,2018-09-27 00:00:00+00:00,2,105.500000,15.000000,0.608000,0.6080,0.500,...,38.12,Low,2,100.00,0.64,44.54,2313,1,Bullish,No major risk events
6612,MGH,9,2011-04-06 00:00:00+00:00,2015-08-29 00:00:00+00:00,6,46.444444,7.000000,0.738300,0.7623,0.333,...,38.24,Low,6,77.78,3.23,44.36,2314,6,Bullish,Litigation: 1
6615,AUMAU,2,2014-10-02 14:39:02+00:00,2014-10-02 15:23:51+00:00,1,93.500000,16.500000,0.500000,0.5000,0.500,...,38.12,Low,1,100.00,0.00,41.69,2317,1,Bullish,No major risk events
6616,TCC,1,2015-01-27 00:00:00+00:00,2015-01-27 00:00:00+00:00,1,67.000000,10.000000,0.500000,0.5000,0.500,...,38.12,Low,1,100.00,0.00,41.69,2317,1,Bullish,No major risk events


## 18. Keyword News Search

In [26]:
def search_news(

    keyword,

    limit=20

):

    keyword = keyword.lower()

    mask = (

        financial_df["headline"]

        .str.lower()

        .str.contains(

            keyword,

            regex=False,

            na=False

        )

    )

    results = financial_df.loc[mask].copy()

    results = results.sort_values(

        "published_date",

        ascending=False

    )

    return results.head(limit)

In [27]:
display(

    search_news(

        "earnings"

    )

)

,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
1030724,1030728,PVH: Q1 Earnings Insights,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-11 20:24:41+00:00,PVH,analyst,2020,6,11,...,25,4,True,PVH: Q1 Earnings Insights,25,4,Earnings,Neutral,1.000,Rule
1332872,1332876,A Look Into Wells Fargo's Price Over Earnings,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:41:22+00:00,WFC,analyst,2020,6,11,...,45,8,True,A Look Into Wells Fargo's Price Over Earnings,45,8,Earnings,Neutral,1.000,Rule
846190,846193,Price Over Earnings Overview: Morgan Stanley,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:03:37+00:00,MS,analyst,2020,6,11,...,44,6,True,Price Over Earnings Overview: Morgan Stanley,44,6,Earnings,Neutral,1.000,Rule
1030730,1030734,"Earnings Scheduled For June 11, 2020",https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-11 08:14:33+00:00,PVH,analyst,2020,6,11,...,36,6,True,"Earnings Scheduled For June 11, 2020",36,6,Earnings,Neutral,1.000,Rule
1228036,1228040,"Earnings Scheduled For June 11, 2020",https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-11 08:14:33+00:00,TNP,analyst,2020,6,11,...,36,6,True,"Earnings Scheduled For June 11, 2020",36,6,Earnings,Neutral,1.000,Rule
1289467,1289471,VBI Vaccines Option Alert: Jul 17 $2.5 Calls S...,https://www.benzinga.com/markets/options/20/06...,Charles Gross,2020-06-10 18:51:32+00:00,VBIV,analyst,2020,6,10,...,135,25,True,VBI Vaccines Option Alert: Jul 17 $2.5 Calls S...,135,25,Options Activity,Neutral,0.733,Rule
530847,530850,A Look Into General Dynamics's Price Over Earn...,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-10 18:30:56+00:00,GD,analyst,2020,6,10,...,50,8,True,A Look Into General Dynamics's Price Over Earn...,50,8,Earnings,Neutral,1.000,Rule
1199008,1199012,A Look Into Technical Communications' Price Ov...,https://www.benzinga.com/markets/penny-stocks/...,Benzinga Insights,2020-06-10 17:46:32+00:00,TCCO,analyst,2020,6,10,...,57,8,True,A Look Into Technical Communications' Price Ov...,57,8,Earnings,Neutral,1.000,Rule
238006,238007,Recap: Chico's FAS Q1 Earnings,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-10 14:46:56+00:00,CHS,analyst,2020,6,10,...,30,5,True,Recap: Chico's FAS Q1 Earnings,30,5,Earnings,Neutral,1.000,Rule
539708,539711,Graham shares are trading lower after the comp...,https://www.benzinga.com/wiim/20/06/16222521/g...,Benzinga Newsdesk,2020-06-10 14:26:20+00:00,GHM,analyst,2020,6,10,...,79,12,True,Graham shares are trading lower after the comp...,79,12,Earnings,Bearish,0.833,Rule


## 19. Universal Search

In [28]:
def search(query):

    query = str(query)

    upper = query.upper()

    lower = query.lower()

    # Company ticker

    if upper in company_lookup:

        print("Company Search\n")

        return search_company(upper)

    # Event

    if lower in event_index:

        print("Event Search\n")

        return search_event(query)

    # Publisher

    if lower in publisher_index:

        print("Publisher Search\n")

        return search_publisher(query)

    # Otherwise search headlines

    print("Keyword Search\n")

    return search_news(query)

## 20. Test Universal Search

In [29]:
# Company

result = search("TSLA")

display_company(result)

Company Search

COMPANY PROFILE


,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,Partnership,Product Launch,Regulatory Approval,Revenue Guidance,Share Buyback,Stock Split,Technical Analysis,Trading Halt,risk_rank,intelligence_rank
0,TSLA,1874,2019-07-01 00:00:00+00:00,2020-06-10 21:02:47+00:00,58,87.258271,13.965848,0.818553,0.7612,0.333,...,11,824,15,15,0,84,19,1,192,712



COMPANY ANALYTICS


,ticker,total_news,first_news,latest_news,unique_publishers,avg_headline_length,avg_word_count,avg_confidence,median_confidence,min_confidence,...,normalized_risk_score,risk_level,event_diversity,market_stability_score,activity_score,composite_score,overall_rank,cluster,company_fingerprint,risk_explanation
0,TSLA,1874,2019-07-01 00:00:00+00:00,2020-06-10 21:02:47+00:00,58,87.258271,13.965848,0.818553,0.7612,0.333,...,40.29,Medium,28,93.28,66.37,79.57,12,4,"Bullish, Diversified",Fraud: 5 | Cybersecurity: 2 | Litigation: 19 |...



SIMILAR COMPANIES


,ticker,similar_company,similarity_score
110,TSLA,NAV,0.9461
111,TSLA,NXPI,0.9433
112,TSLA,TTM,0.9417
113,TSLA,NOK,0.9357
114,TSLA,UA,0.9349
115,TSLA,HMC,0.9348
116,TSLA,TM,0.9336
117,TSLA,NE,0.9333
118,TSLA,W,0.9326
119,TSLA,KBH,0.9315



GRAPH NEIGHBORS
['NAV' 'NXPI' 'TTM' 'NOK' 'UA' 'HMC' 'TM' 'NE' 'W' 'KBH']

LATEST TIMELINE


,news_id,ticker,published_date,headline,publisher,final_event,market_signal,final_confidence,event_number,previous_event_date,...,bullish_event,bearish_event,bullish_events_so_far,bearish_events_so_far,event_importance,event_severity,event_date,daily_event_count,activity_level,is_latest_event
2857678,1245072,TSLA,2020-06-10 21:02:47+00:00,Tesla's Stock Closes At All-Time High As Musk ...,Drew Levine,Stock Split,Neutral,0.7389,1874,2020-06-10 19:08:09+00:00,...,0,0,157,49,4,Medium,2020-06-10,10,High,True
2857677,1245073,TSLA,2020-06-10 19:08:09+00:00,'Tesla factory workplace safety is 5% better t...,Benzinga Newsdesk,Product Launch,Neutral,0.7136,1873,2020-06-10 16:41:58+00:00,...,0,0,157,49,7,High,2020-06-10,10,High,False
2857676,1245074,TSLA,2020-06-10 16:41:58+00:00,'Tesla hacker unlocks Performance upgrade and ...,Benzinga Newsdesk,Analyst Rating,Bullish,1.0000,1872,2020-06-10 15:33:18+00:00,...,1,0,157,49,5,Medium,2020-06-10,10,High,False
2857675,1245075,TSLA,2020-06-10 15:33:18+00:00,GM On Track To Spend $20B On EV And AV Develop...,Benzinga Newsdesk,Product Launch,Neutral,0.7237,1871,2020-06-10 14:15:07+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857674,1245076,TSLA,2020-06-10 14:15:07+00:00,"Tesla's Journey To $1,000 In 2020",Wayne Duggan,Product Launch,Neutral,0.7711,1870,2020-06-10 13:58:54+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857673,1245077,TSLA,2020-06-10 13:58:54+00:00,Tesla Shares Mark Session And New All-Time Hig...,Benzinga Newsdesk,Product Launch,Neutral,0.7460,1869,2020-06-10 13:00:24+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857672,1245078,TSLA,2020-06-10 13:00:24+00:00,Wedbush Says Tesla Has 'Game Changing' Develop...,Jayson Derrick,Product Launch,Neutral,0.7269,1868,2020-06-10 12:43:13+00:00,...,0,0,156,49,7,High,2020-06-10,10,High,False
2857671,1245079,TSLA,2020-06-10 12:43:13+00:00,"Wedbush Maintains Neutral on Tesla, Raises Pri...",Benzinga Newsdesk,Analyst Rating,Bullish,1.0000,1867,2020-06-10 12:14:33+00:00,...,1,0,156,49,5,Medium,2020-06-10,10,High,False
2857670,1245080,TSLA,2020-06-10 12:14:33+00:00,Musk Says It's Time To Ramp Production Of Tesl...,Tanzeel Akhtar,Product Launch,Neutral,0.7728,1866,2020-06-10 11:55:49+00:00,...,0,0,155,49,7,High,2020-06-10,10,High,False
2857669,1245081,TSLA,2020-06-10 11:55:49+00:00,Tesla shares are trading higher after Wedbush ...,Benzinga Newsdesk,Analyst Rating,Bullish,0.8000,1865,2020-06-02 00:00:00+00:00,...,1,0,155,49,5,Medium,2020-06-10,10,High,False


In [30]:
# Event

summary, data = search("Earnings")

print(summary)

display(data.head())

Event Search

{'event': 'earnings', 'articles': 567435, 'companies': 5694, 'publishers': 396, 'avg_confidence': np.float64(0.9441)}


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
1030723,1030727,PVH shares are trading lower after the company...,https://www.benzinga.com/wiim/20/06/16237694/p...,Benzinga Newsdesk,2020-06-11 20:25:21+00:00,PVH,analyst,2020,6,11,...,101,15,True,PVH shares are trading lower after the company...,101,15,Earnings,Bearish,0.833,Rule
1030724,1030728,PVH: Q1 Earnings Insights,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Insights,2020-06-11 20:24:41+00:00,PVH,analyst,2020,6,11,...,25,4,True,PVH: Q1 Earnings Insights,25,4,Earnings,Neutral,1.000,Rule
1030725,1030729,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Newsdesk,2020-06-11 20:15:38+00:00,PVH,analyst,2020,6,11,...,82,13,True,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,82,13,Earnings,Bearish,1.000,Rule
1332872,1332876,A Look Into Wells Fargo's Price Over Earnings,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:41:22+00:00,WFC,analyst,2020,6,11,...,45,8,True,A Look Into Wells Fargo's Price Over Earnings,45,8,Earnings,Neutral,1.000,Rule
846190,846193,Price Over Earnings Overview: Morgan Stanley,https://www.benzinga.com/intraday-update/20/06...,Benzinga Insights,2020-06-11 15:03:37+00:00,MS,analyst,2020,6,11,...,44,6,True,Price Over Earnings Overview: Morgan Stanley,44,6,Earnings,Neutral,1.000,Rule


In [31]:
# Publisher

summary, data = search("Benzinga Newsdesk")

print(summary)

display(data.head())

Publisher Search

{'publisher': 'benzinga newsdesk', 'total_articles': 150392, 'companies': 4049, 'events': 30, 'avg_confidence': np.float64(0.8482)}


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
972643,972647,PG&E Corp Reports CPUC Approved Co.'s Microgri...,https://www.benzinga.com/news/20/06/16238398/p...,Benzinga Newsdesk,2020-06-11 21:11:20+00:00,PCG,analyst,2020,6,11,...,57,8,True,PG&E Corp Reports CPUC Approved Co.'s Microgri...,57,8,Regulatory Approval,Bullish,1.0000,Rule
1258838,1258842,"Twitter Removes About 174,000 China-Linked Acc...",https://www.benzinga.com/tech/20/06/16238284/t...,Benzinga Newsdesk,2020-06-11 21:01:39+00:00,TWTR,analyst,2020,6,11,...,130,18,True,"Twitter Removes About 174,000 China-Linked Acc...",130,18,Other,Neutral,0.6933,Unknown
1347859,1347863,W. P. Carey Inc. Increases Quarterly Dividend ...,https://www.benzinga.com/news/20/06/16237774/w...,Benzinga Newsdesk,2020-06-11 20:30:31+00:00,WPC,analyst,2020,6,11,...,61,9,True,W. P. Carey Inc. Increases Quarterly Dividend ...,61,9,Dividend,Bullish,0.6670,Rule
1030723,1030727,PVH shares are trading lower after the company...,https://www.benzinga.com/wiim/20/06/16237694/p...,Benzinga Newsdesk,2020-06-11 20:25:21+00:00,PVH,analyst,2020,6,11,...,101,15,True,PVH shares are trading lower after the company...,101,15,Earnings,Bearish,0.8330,Rule
1030725,1030729,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,https://www.benzinga.com/news/earnings/20/06/1...,Benzinga Newsdesk,2020-06-11 20:15:38+00:00,PVH,analyst,2020,6,11,...,82,13,True,PVH Q1 Adj. EPS $(3.03) Misses $(1.28) Estimat...,82,13,Earnings,Bearish,1.0000,Rule


In [32]:
# Keyword

display(

    search("inflation")

)

Keyword Search



,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,...,headline_length,word_count,has_time,original_headline,clean_character_count,clean_word_count,final_event,market_signal,final_confidence,classification_source
1094150,1094154,Ruth's Hospitality Group Inc Did Experience Be...,https://www.benzinga.com/news/20/06/16178579/r...,Benzinga Newsdesk,2020-06-04 12:09:59+00:00,RUTH,analyst,2020,6,4,...,194,33,True,Ruth's Hospitality Group Inc Did Experience Be...,194,33,Revenue Guidance,Neutral,0.5000,Rule
2970266,2970285,"Yes, This Is A Bullish Setup In Inflation Expe...",https://talkmarkets.com/content/yes-this-is-a-...,TalkMarkets,2020-05-31 00:00:00+00:00,TIP,partner,2020,5,31,...,54,9,False,"Yes, This Is A Bullish Setup In Inflation Expe...",54,9,Economic Data,Neutral,0.5000,Rule
1399972,1399976,First the Deflationary Deluge of Assets Crashi...,https://talkmarkets.com/content/first-the-defl...,TalkMarkets,2020-05-31 00:00:00+00:00,AAPL,partner,2020,5,31,...,79,12,False,First the Deflationary Deluge of Assets Crashi...,79,12,Economic Data,Neutral,1.0000,Rule
3008850,3008869,"Deflation, Inflation, Or Stagflation?",https://talkmarkets.com/content/deflation-infl...,TalkMarkets,2020-05-25 00:00:00+00:00,TSN,partner,2020,5,25,...,37,4,False,"Deflation, Inflation, Or Stagflation?",37,4,Economic Data,Neutral,1.0000,Rule
2858966,2858983,Silver & Inflation Expectations,https://talkmarkets.com/content/silver--inflat...,TalkMarkets,2020-05-19 00:00:00+00:00,SLV,partner,2020,5,19,...,31,4,False,Silver & Inflation Expectations,31,4,Economic Data,Neutral,0.5000,Rule
2858970,2858987,"Spiking Food Prices, Kick Off Inflationary End...",https://talkmarkets.com/content/spiking-food-p...,TalkMarkets,2020-05-19 00:00:00+00:00,SLV,partner,2020,5,19,...,50,7,False,"Spiking Food Prices, Kick Off Inflationary End...",50,7,Economic Data,Neutral,0.7431,Semantic
546932,546935,Fed's Bullard Says Doesn't See Much Role For G...,https://www.benzinga.com/markets/commodities/2...,Benzinga Newsdesk,2020-05-12 00:00:00+00:00,GLD,analyst,2020,5,12,...,76,13,True,Fed's Bullard Says Doesn't See Much Role For G...,76,13,Monetary Policy,Neutral,0.3330,Rule
2859080,2859097,Corona Vs 2008 The Difference Is Inflation,https://talkmarkets.com/content/corona-vs-2008...,TalkMarkets,2020-04-28 00:00:00+00:00,SLV,partner,2020,4,28,...,42,7,False,Corona Vs 2008 The Difference Is Inflation,42,7,Economic Data,Neutral,1.0000,Rule
2838941,2838958,Corona Vs 2008 The Difference Is Inflation,https://talkmarkets.com/content/corona-vs-2008...,TalkMarkets,2020-04-28 00:00:00+00:00,SIL,partner,2020,4,28,...,42,7,False,Corona Vs 2008 The Difference Is Inflation,42,7,Economic Data,Neutral,1.0000,Rule
2045089,2045097,AUD/USD Forecast April 27-May 1 – Will Consume...,https://talkmarkets.com/content/audusd-forecas...,TalkMarkets,2020-04-26 00:00:00+00:00,FXA,partner,2020,4,26,...,86,14,False,AUD/USD Forecast April 27-May 1 – Will Consume...,86,14,Earnings,Neutral,0.3330,Rule


## 21. Save Search Indexes

In [33]:
search_objects = {

    "company_lookup.pkl": company_lookup,
    "analytics_lookup.pkl": analytics_lookup,
    "timeline_lookup.pkl": timeline_lookup,
    "similarity_lookup.pkl": similarity_lookup,
    "neighbor_lookup.pkl": neighbor_lookup,
    "company_index.pkl": company_index,
    "publisher_index.pkl": publisher_index,
    "event_index.pkl": event_index,
    "cluster_index.pkl": cluster_index,
    "fingerprint_index.pkl": fingerprint_index,
    "news_index.pkl": news_index

}

for filename, obj in search_objects.items():

    with open(

        os.path.join(SEARCH_PATH, filename),

        "wb"

    ) as f:

        pickle.dump(obj, f)

print("Search indexes saved successfully.")

print(f"Total Objects Saved : {len(search_objects)}")
print(f"Output Folder       : {SEARCH_PATH}")

Search indexes saved successfully.
Total Objects Saved : 11
Output Folder       : /content/drive/MyDrive/FinSight_AI/data/processed/search


## 22. Performance Evaluation

In [38]:
import time

tests = [

    ("Company", lambda: search_company("TSLA")),
    ("Timeline", lambda: search_timeline("TSLA")),
    ("Similar", lambda: search_similar_companies("TSLA")),
    ("Neighbors", lambda: search_neighbors("TSLA")),
    ("Keyword", lambda: search_news("earnings"))

]

performance = []

for name, func in tests:

    start = time.perf_counter()

    func()

    end = time.perf_counter()

    performance.append({

        "Search Type": name,

        "Time (ms)": round(

            (end-start)*1000,

            3

        )

    })

performance_df = pd.DataFrame(performance)

display(performance_df)

,Search Type,Time (ms)
0,Company,4.013
1,Timeline,1.323
2,Similar,0.005
3,Neighbors,32.755
4,Keyword,2713.572


## 23. Demonstration Queries

In [35]:
example_queries = [

    "TSLA",

    "AAPL",

    "MSFT",

    "Earnings",

    "Product Launch",

    "Benzinga Newsdesk",

    "inflation",

    "cyber attack"

]

print("Example Search Queries")

for q in example_queries:

    print(q)

Example Search Queries
TSLA
AAPL
MSFT
Earnings
Product Launch
Benzinga Newsdesk
inflation
cyber attack


## 24. Validate Saved Objects

In [36]:
saved_files = sorted(

    os.listdir(

        SEARCH_PATH

    )

)

validation = pd.DataFrame({

    "Saved File": saved_files

})

display(validation)

,Saved File
0,analytics_lookup.pkl
1,cluster_index.pkl
2,company_index.pkl
3,company_lookup.pkl
4,event_index.pkl
5,fingerprint_index.pkl
6,neighbor_lookup.pkl
7,news_index.pkl
8,publisher_index.pkl
9,similarity_lookup.pkl


## 25. Summary

In [37]:
summary = pd.DataFrame({

    "Metric":[

        "Financial News",

        "Companies",

        "Publishers",

        "Events",

        "Knowledge Graph Nodes",

        "Knowledge Graph Edges",

        "Search Objects",

        "Example Queries"

    ],

    "Value":[

        len(financial_df),

        len(company_lookup),

        len(publisher_index),

        len(event_index),

        len(nodes),

        len(edges),

        len(search_objects),

        len(example_queries)

    ]

})

display(summary)

,Metric,Value
0,Financial News,3215296
1,Companies,6619
2,Publishers,1047
3,Events,30
4,Knowledge Graph Nodes,7720
5,Knowledge Graph Edges,427023
6,Search Objects,11
7,Example Queries,8
